In [1]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch
import pandas as pd
import numpy as np
device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
from pathlib import Path
DATA_PATH = Path("..")/"Dataset/test.csv"
# Change this if you want to test with different dataset for MNIST Classfication

In [3]:
class MNISTDataset(Dataset):
    def __init__(self, dataframe, transform=None, is_test=False):
        if is_test:
            self.images = dataframe.values.reshape(-1, 28, 28).astype(np.float32) / 255.0
            self.labels = None
        else:
            self.labels = dataframe.iloc[:, 0].values
            self.images = dataframe.iloc[:, 1:].values.reshape(-1, 28, 28).astype(np.float32) / 255.0
        
        self.transform = transform
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx] if self.labels is not None else -1
        
        image = torch.tensor(image).unsqueeze(0)
        
        if self.transform:
            image = self.transform(image)
        
        if self.labels is not None:
            return image, torch.tensor(label, dtype=torch.long)
        return image


class DeepCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 3 * 3, 256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        return self.fc(self.conv(x))

In [4]:
# Load model (same architecture)
model = DeepCNN()

model_path = Path("..") / "outputs" / "best_model.pth"

model.load_state_dict(torch.load(model_path, weights_only=True))
model.eval()

DeepCNN(
  (conv): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=1152, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.25, inplace=False)
    (4): Linear(in_features=256, out_features=10, bias=True)
  )
)

In [5]:
# Load new data you want, here I take the data from test csv
new_data = pd.read_csv(DATA_PATH)  # Replace with your new CSV
new_dataset = MNISTDataset(new_data, transform=None, is_test=True)
new_loader = DataLoader(new_dataset, batch_size=64, shuffle=False)

# === Make predictions ===
predictions = []

with torch.no_grad():
    for images in new_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(1).cpu().numpy()
        predictions.extend(preds)

# === Save predictions ===
submission = pd.DataFrame({
    'ImageId': range(1, len(predictions) + 1),
    'Label': predictions
})
output_path = Path("..")/ "outputs"/"submission.csv"
submission.to_csv(output_path, index=False)

print(f"Predictions saved! Shape: {submission.shape}")
print(submission.head(10))

Predictions saved! Shape: (28000, 2)
   ImageId  Label
0        1      2
1        2      0
2        3      9
3        4      9
4        5      3
5        6      7
6        7      0
7        8      3
8        9      0
9       10      3
